In [8]:
import pandas as pd #PANDAS
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import statistics
import numpy as np
import sys #GETTING EQUATIONS
import os
sys.path.append('../../../bgc-analysis')
from  calc_sensor import pH_sbe, o2_sbe, mcoms
from datetime import datetime
from db_api import get_continuous_one_profile,get_continuous_one_float
from calc_sensor import pH_sbe
sys.path.clear
sys.path.append('C:/Users/athay/Documents/Repositories/test-analysis')
from get_cal_sn import get_cal
sys.path.clear
sys.path.append('C:/Users/athay/Documents/Repositories/bgc-testing/flowthrough_testing')
from flowthrough_analysis import diff_analysis

In [9]:
# Input the date 
date = "18-Feb-2026"
platformtype = "&PLATFORM_TYPE=SOLO_BGC_MRV"

In [10]:
folderpath = 'Z:\\www\\bgc\\pressure_test\\BGC_SOLO\\'+date
try:
    os.mkdir(folderpath)
except:
    pass


In [11]:
# Read Table
data = pd.read_excel(f"G:/Shared drives/WHOI Argo/BGC/Float Tracking/pressure_test_data/{date}.xlsx")
data['fid'] =data['fid'].astype('category')
data.rename(columns={'fid':'Float Id','p':'Pressure (dbar)','t':'Temperature (C)','s':'Salinity (PSU)','no3':'NO3 (uM)','time':'Time'},inplace = True)

In [12]:
for sn in data['Float Id'].unique():
    sns = str(sn)
    cal = get_cal(sn,platformtype)
    sub = data.loc[data['Float Id'] == sn,:]
    #PH
    data.loc[data['Float Id'] == sn, 'PH_TOTAL'] = pH_sbe(cal['TRANSISTOR_PH']['PREDEPLOYMENT_CALIB_COEFFICIENT'],sub['phvrs'], sub['Pressure (dbar)'], sub['Temperature (C)'], sub['Salinity (PSU)'])[0]
    #Oxygen
    data.loc[data['Float Id'] == sn, 'DOXY'] = (o2_sbe(cal['OPTODE_DOXY']['PREDEPLOYMENT_CALIB_COEFFICIENT'], sub['doxyphasedelay'], sub['doxytempvolts'], sub['Pressure (dbar)'], sub['Temperature (C)'], sub['Salinity (PSU)'])[0])


In [13]:
params = ['Pressure (dbar)','Temperature (C)','Salinity (PSU)','doxyphasedelay','doxytempvolts','phvrs','phvk','PH_TOTAL','DOXY']

In [14]:
for i in params:
    fig = px.scatter(data,x = data['Time'], y = data[i], color = data['Float Id'])
    fig.write_html(folderpath +'\\' + str(i) + ".html")

In [15]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x = data['Time'],
    y = data['phvk'],
    name = "pH vk",
    yaxis = "y1",
))
fig.add_trace(go.Scatter(
    x = data['Time'],
    y = data['PH_TOTAL'],
    yaxis = "y2",
    name = "pH Total",
))
fig.add_trace(go.Scatter(
    x = data['Time'],
    y = data['phvrs'],
    yaxis = "y3",
    name = "pH vrs",
))
fig.add_trace(go.Scatter(
    x = data['Time'],
    y = data['Pressure (dbar)'],
    yaxis = "y4",
    name = "Pressure",
))
fig.update_layout(
    xaxis = dict(domain=[0.11,0.85]),
    yaxis = dict(title=dict(text = "pH vk",font=dict(color="blue")),tickfont=dict(color="blue"),side="left",position =0),
    yaxis2 = dict(title=dict(text = "pH Total",font=dict(color="red")),tickfont=dict(color="red"),anchor="free", overlaying="y",side="right",position = .85),
    yaxis3 = dict(title=dict(text = "pH vrs",font=dict(color="green")),tickfont=dict(color="green"),anchor="free",overlaying="y",side="left",position = 0.11),   
    yaxis4 = dict(title=dict(text = "Pressure",font=dict(color="purple")),tickfont=dict(color="purple"),anchor="free",overlaying="y",side="right",position = .95))
fig.write_html(folderpath +'\\' + "pH_comparisons.html")

fig.show()